In [2]:
# notebooks/02_Pipeline_A_2D-CNN_Log-Mel.ipynb
# --- 1. SETUP E CONFIGURAZIONE (CORRETTO PER GOOGLE COLAB) ---
import os
import sys
import json
import glob
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

import tensorflow as tf
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())


# --- MODIFICHE PER COLAB ---

# 1. Monta Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True) # Aggiungo force_remount per sicurezza


# Definisci i percorsi conosciuti
known_paths = [
    '/content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab', # Percorso luke
    '/content/drive/MyDrive/Colab/urbanSmartSound_Colab'   # Percorso mirko
]


# 2. Definisci il percorso radice del tuo progetto su Google Drive
#PROJECT_ROOT = '/content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab'

# 3. Aggiungi la cartella del progetto (che contiene 'src') al path di Python
#if PROJECT_ROOT not in sys.path:
    #sys.path.append(PROJECT_ROOT)


# Prova ogni percorso conosciuto finché non ne trovi uno valido
PROJECT_ROOT = None
for path in known_paths:
    if os.path.exists(path):
        PROJECT_ROOT = path
        break

if PROJECT_ROOT:
    print(f"Trovata cartella di progetto valida a: {PROJECT_ROOT}")
    os.chdir(PROJECT_ROOT)
    sys.path.append(PROJECT_ROOT)
    !ls
else:
    print("ERRORE: Nessuno dei percorsi conosciuti è valido.")
    # Come fallback, chiedi all'utente di inserirlo manualmente
    PROJECT_ROOT = input("Per favore, incolla il percorso completo alla cartella 'urbanSmartSound_Colab': ")
    if os.path.exists(PROJECT_ROOT):
        os.chdir(PROJECT_ROOT)
        sys.path.append(PROJECT_ROOT)
        !ls
    else:
        print("Il percorso inserito non è valido.")




# Ora puoi importare i tuoi moduli
from src import data_loader, models, evaluation

# --- FINE MODIFICHE PER COLAB ---

# Configurazione specifica della pipeline
PIPELINE_NAME = "Pipeline_A_Log-Mel"
FEATURE_KEY = 'log_mel_spec'
FEATURE_LOADER_FN = data_loader.load_feature
MODEL_CREATOR_FN = models.create_2d_cnn_gru_model
MODEL_FILENAME = "model_A_logmel.keras"
METADATA_FILENAME = "model_A_logmel_metadata.json"

# === CORREZIONE DEFINITIVA DI TUTTI I PERCORSI ===
# Usa os.path.join con PROJECT_ROOT per creare percorsi assoluti e robusti
FEATURES_DIR = os.path.join(PROJECT_ROOT, "data/processed/features_v2")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
CSV_PATH = os.path.join(PROJECT_ROOT, "data/raw/UrbanSound8K.csv")
# ================================================

os.makedirs(MODELS_DIR, exist_ok=True)

print(f"--- ESECUZIONE: {PIPELINE_NAME} ---")
print(f"Project Root: {PROJECT_ROOT}")
print(f"Features Directory: {FEATURES_DIR}") # Ora stamperà il percorso completo
print(f"Models Directory: {MODELS_DIR}")
print(f"CSV File Path: {CSV_PATH}")


# --- 2. CARICAMENTO DATI ---
print("\n[Fase 1/5] Caricamento di tutti i dati...")
all_data = {}
fold_dirs = sorted(glob.glob(os.path.join(FEATURES_DIR, "fold*")))
for fold_dir in fold_dirs:
    fold_name = os.path.basename(fold_dir)
    print(f"Caricando {fold_name}...")
    X_fold, y_fold = data_loader.collect_fold_data(
        fold_dir, FEATURE_LOADER_FN, feature_key=FEATURE_KEY
    )
    all_data[fold_name] = (X_fold, y_fold)
print("Caricamento completato.")


[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 2285620892227819342
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 40419328000
locality {
  bus_id: 1
  links {
  }
}
incarnation: 12832023715894857276
physical_device_desc: "device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:00:04.0, compute capability: 8.0"
xla_global_id: 416903419
]
Mounted at /content/drive
Trovata cartella di progetto valida a: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab
data  models  notebooks  src
--- ESECUZIONE: Pipeline_A_Log-Mel ---
Project Root: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab
Features Directory: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab/data/processed/features_v2
Models Directory: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab/models
CSV File Path: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab/data/raw/UrbanSound8K.csv

[Fase 1/5] Caricament

In [6]:

# --- 3. CROSS-VALIDATION ---
print("\n[Fase 2/5] Avvio Cross-Validation...")
class_names = data_loader.get_class_map(CSV_PATH)

num_classes = len(class_names)
input_shape = list(all_data.values())[0][0][0].shape

# ... il resto del codice della cella non cambia ...
fold_accuracies = []
all_y_true_cv, all_y_pred_cv = [], []

for i, val_fold_name in enumerate(all_data.keys()):
    print(f"\n--- CV Fold {i+1}/{len(all_data)} (Validation: {val_fold_name}) ---")

    # Preparazione dati train/val per questo fold
    X_val, y_val = all_data[val_fold_name]
    train_folds = [data for name, data in all_data.items() if name != val_fold_name]
    X_train = np.vstack([f[0] for f in train_folds])
    y_train = np.concatenate([f[1] for f in train_folds])

    # Creazione e training del modello
    model = MODEL_CREATOR_FN(input_shape, num_classes)
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,  # Aumentato, EarlyStopping gestirà l'arresto
        batch_size=32,
        #callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
        verbose=1
    )

    # Valutazione
    _, acc = model.evaluate(X_val, y_val, verbose=0)
    fold_accuracies.append(acc)
    y_pred = np.argmax(model.predict(X_val), axis=1)
    all_y_true_cv.extend(y_val)
    all_y_pred_cv.extend(y_pred)
    print(f"Accuracy del fold: {acc:.4f}")

mean_acc_cv = np.mean(fold_accuracies)
std_acc_cv = np.std(fold_accuracies)
print(f"\nAccuracy media CV: {mean_acc_cv:.4f} ± {std_acc_cv:.4f}")




[Fase 2/5] Avvio Cross-Validation...

--- CV Fold 1/10 (Validation: fold1) ---
Epoch 1/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - accuracy: 0.3322 - loss: 2.0209 - val_accuracy: 0.4613 - val_loss: 1.6126
Epoch 2/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.5788 - loss: 1.2351 - val_accuracy: 0.5607 - val_loss: 1.2385
Epoch 3/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.6913 - loss: 0.9005 - val_accuracy: 0.6224 - val_loss: 0.9836
Epoch 4/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.7574 - loss: 0.7149 - val_accuracy: 0.5638 - val_loss: 1.1443
Epoch 5/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.8035 - loss: 0.6148 - val_accuracy: 0.6056 - val_loss: 1.0855
Epoch 6/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.8359 - loss: 0.5005 - val_accuracy: 0.5743 - val_loss: 1.1671
Epoch 7/50
281/281 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.8579 - loss: 0.4304 - val_accuracy: 0.5858 - val_loss: 1.2859
Epoch 8/50
281/28

In [8]:
# --- 4. ADDESTRAMENTO MODELLO FINALE ---
print("\n[Fase 3/5] Addestramento del modello finale su split 80/20...")
X_train_final, y_train_final, X_test_final, y_test_final = data_loader.get_train_test_split_from_folds(
    all_data,
    meta_file_path=CSV_PATH,  # Passiamo il percorso corretto!
    test_size=0.2             # Questo è lo split 80/20 che volevi
)

# Calcolo pesi per classi sbilanciate
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_final), y=y_train_final)
class_weight_dict = dict(enumerate(class_weights))

final_model = MODEL_CREATOR_FN(input_shape, num_classes)
final_history = final_model.fit(
    X_train_final, y_train_final,
    validation_data=(X_test_final, y_test_final),
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[
        #EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)
    ],
    verbose=1
)



[Fase 3/5] Addestramento del modello finale su split 80/20...
Epoch 1/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - accuracy: 0.3239 - loss: 2.0637 - val_accuracy: 0.4472 - val_loss: 1.6350 - learning_rate: 5.0000e-04
Epoch 2/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.5455 - loss: 1.3418 - val_accuracy: 0.5017 - val_loss: 1.5086 - learning_rate: 5.0000e-04
Epoch 3/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.6625 - loss: 1.0215 - val_accuracy: 0.6096 - val_loss: 1.0719 - learning_rate: 5.0000e-04
Epoch 4/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.7306 - loss: 0.7883 - val_accuracy: 0.6502 - val_loss: 1.0139 - learning_rate: 5.0000e-04
Epoch 5/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.7770 - loss: 0.6667 - val_accuracy: 0.6151 - val_loss: 1.3147 - learning_rate: 5.0000e-04
Epoch 6/100
217/217 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.8284 - loss: 0.5253 - val_accuracy: 0.7036 - val_loss: 0.9099 - learning_rate:

In [9]:

# --- 5. VALUTAZIONE E SALVATAGGIO ---
print("\n[Fase 4/5] Valutazione del modello finale...")
final_loss, final_accuracy = final_model.evaluate(X_test_final, y_test_final, verbose=0)
print(f"Performance finale sul Test Set:")
print(f"  - Loss: {final_loss:.4f}")
print(f"  - Accuracy: {final_accuracy:.4f}\n")

print("\n[Fase 5/5] Salvataggio del modello e dei metadati...")
# Salva modello
model_path = os.path.join(MODELS_DIR, MODEL_FILENAME)
final_model.save(model_path)
print(f"Modello salvato in: {model_path}")

# Salva metadati
metadata = {
    "pipeline_name": PIPELINE_NAME,
    "feature_key": FEATURE_KEY,
    "model_filename": MODEL_FILENAME,
    "input_shape": input_shape,
    "num_classes": num_classes,
    "class_names": class_names,
    "cv_performance": {"mean_accuracy": mean_acc_cv, "std_accuracy": std_acc_cv},
    "final_test_performance": {"accuracy": final_accuracy, "loss": final_loss}
}
metadata_path = os.path.join(MODELS_DIR, METADATA_FILENAME)
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)
print(f"Metadati salvati in: {metadata_path}")


[Fase 4/5] Valutazione del modello finale...
Performance finale sul Test Set:
  - Loss: 0.5683
  - Accuracy: 0.8471


[Fase 5/5] Salvataggio del modello e dei metadati...
Modello salvato in: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab/models/model_A_logmel.keras
Metadati salvati in: /content/drive/MyDrive/AI_FETA/Project/urbanSmartSound_Colab/models/model_A_logmel_metadata.json
